# 02 · SECOM exploratory analysis

**Goal:** understand 590 anonymised process measurements before predicting pass/fail.

**Questions**
1. How imbalanced is pass vs fail?
2. How much data is missing, and which features carry no information?
3. How redundant are the sensors?
4. Does the fail rate drift over time? (decides random vs time-based split)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from wafer import eda
from wafer.data import load_secom, load_wm811k

FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 3)

In [ ]:
X, meta = load_secom()
print(X.shape)
meta['fail'].value_counts()

## 1. Class balance

In [ ]:
print(f"fail rate: {meta['fail'].mean():.1%}")

## 2. Missing values and uninformative features

In [ ]:
eda.missing_profile(X)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(X.isna().mean(), bins=50)
ax.set_xlabel('fraction missing'); ax.set_ylabel('features')
ax.set_title('Missing rate per feature'); plt.tight_layout()
plt.savefig(FIG_DIR / 'secom_missing.png', dpi=150)

In [ ]:
low = eda.low_information_features(X)
{k: len(v) for k, v in low.items()}

## 3. Redundant sensors

In [ ]:
pairs = eda.correlated_pairs(X.drop(columns=low['constant']), threshold=0.95)
print(f'{len(pairs)} feature pairs with |r| > 0.95')
pairs.head(10)

## 4. Fail rate over time

In [ ]:
weekly = eda.fail_rate_over_time(meta)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(weekly.index, weekly['runs'], width=5, alpha=0.3, label='runs')
ax2 = ax1.twinx(); ax2.plot(weekly.index, weekly['fail_rate'], 'r-o', label='fail rate')
ax1.set_ylabel('runs per week'); ax2.set_ylabel('fail rate')
ax1.set_title('SECOM runs and fail rate per week'); plt.tight_layout()
plt.savefig(FIG_DIR / 'secom_fail_rate_time.png', dpi=150)

## Findings

_Fill in after running on the real data._

- Fail rate: …
- Features dropped (constant / >50% missing): …
- Highly correlated pairs: …
- Drift over time: …

**Decisions for step 3:** imputation, feature reduction, split strategy, metric (PR-AUC).